In [1]:
from copy import deepcopy

target = 3

initial = {
    "inventory": {"folders": 3, "labels": 2, "spares": 1},
    "packed": 0,
    "verified": False,
}

def goal_met(state, target):
    return state["packed"] == target and state["verified"]

print(initial)
print("Goal met:", goal_met(initial, target))

{'inventory': {'folders': 3, 'labels': 2, 'spares': 1}, 'packed': 0, 'verified': False}
Goal met: False


Three kits cannot be assembled yet because there are only two ready labels. The goal is not met because no kits have been packed or verified.

In [2]:
first_plan = [
    {"action": "inspect", "quantity": 0},
    {"action": "assemble", "quantity": 3},
    {"action": "check", "quantity": 0},
]

for number, proposal in enumerate(first_plan, start=1):
    print(number, proposal)

print("State still unchanged:", initial["packed"] == 0)

1 {'action': 'inspect', 'quantity': 0}
2 {'action': 'assemble', 'quantity': 3}
3 {'action': 'check', 'quantity': 0}
State still unchanged: True


The plan is missing a restock step. Checking before assembly would fail. The plan lists the actions but does not change the inventory.

In [3]:
def inspect_stock(state, quantity, target):
    return "Stock inspected."

def restock(state, quantity, target):
    stock = state["inventory"]
    if stock["spares"] < quantity:
        raise ValueError("not enough spare labels")
    stock["spares"] -= quantity
    stock["labels"] += quantity
    state["verified"] = False
    return f"Moved {quantity} spare label(s)."

demo = deepcopy(initial)
print(restock(demo, 1, target))
print(demo["inventory"])
print("Original labels:", initial["inventory"]["labels"])

Moved 1 spare label(s).
{'folders': 3, 'labels': 3, 'spares': 0}
Original labels: 2


If two spare labels are requested, nothing should move because only one is available. Restocking the copied inventory does not change the original inventory.

In [4]:
def assemble(state, quantity, target):
    stock = state["inventory"]
    if state["packed"] + quantity > target:
        raise ValueError("would exceed the target")
    if stock["folders"] < quantity or stock["labels"] < quantity:
        raise ValueError("not enough ready materials")
    stock["folders"] -= quantity
    stock["labels"] -= quantity
    state["packed"] += quantity
    state["verified"] = False
    return f"Assembled {quantity} kit(s)."

def check_kits(state, quantity, target):
    state["verified"] = state["packed"] == target
    return f"Count check: {state['verified']}"

print(assemble(demo, 3, target))
print("Before check:", goal_met(demo, target))
print(check_kits(demo, 0, target))
print("After check:", goal_met(demo, target))

Assembled 3 kit(s).
Before check: False
Count check: True
After check: True


Assembly would fail with the original inventory because there are not enough labels. Even after packing three kits, the goal is not complete until they are verified.

In [5]:
tools = {
    "inspect": inspect_stock,
    "restock": restock,
    "assemble": assemble,
    "check": check_kits
}

def execute(proposal, state, target):
    try:
        if type(proposal) is not dict or set(proposal) != {"action", "quantity"}:
            raise ValueError("expected action and quantity")

        name, q = proposal["action"], proposal["quantity"]

        if type(name) is not str or name not in tools:
            raise ValueError("unknown action")

        if type(q) is not int:
            raise ValueError("quantity must be an integer")

        if name in {"inspect", "check"}:
            if q != 0:
                raise ValueError("read/check quantity must be zero")
        elif not 1 <= q <= target:
            raise ValueError("quantity outside task bounds")

        note = tools[name](state, q, target)
        ok = True

    except ValueError as error:
        note, ok = str(error), False

    return {"ok": ok, "note": note, "state": deepcopy(state)}

print(execute({"action": "announce_success", "quantity": 0},
              deepcopy(initial), target)["note"])

unknown action


An unknown action or a string quantity should be rejected. Validation prevents invalid actions from changing the inventory.

In [6]:
def follow_plan(plan, start, target):
    state = deepcopy(start)
    trace = []

    for proposal in plan:
        observation = execute(proposal, state, target)
        trace.append({"proposal": deepcopy(proposal),
                      "observation": observation})

        if not observation["ok"]:
            break

    return state, trace

failed_state, failed_trace = follow_plan(first_plan, initial, target)

for event in failed_trace:
    print(event["proposal"]["action"], event["observation"]["note"])

print("Packed:", failed_state["packed"])
print("Goal met:", goal_met(failed_state, target))

inspect Stock inspected.
assemble not enough ready materials
Packed: 0
Goal met: False


The plan should stop after two attempts. Assembly fails because there are not enough labels, so no kits are packed and the check never runs.

In [7]:
feedback = {
    "evidence": failed_trace[-1]["observation"]["note"],
    "cause": "The plan tried three kits with two ready labels.",
    "change": "Inspect stock, then restock the one-label shortage.",
}

lessons = [feedback]

revised_plan = [
    first_plan[0],
    {"action": "restock", "quantity": 1},
    first_plan[1],
    first_plan[2]
]

revised_state, revised_trace = follow_plan(revised_plan, initial, target)

print(feedback["change"])
print([e["proposal"]["action"] for e in revised_trace])
print("Goal met:", goal_met(revised_state, target))

Inspect stock, then restock the one-label shortage.
['inspect', 'restock', 'assemble', 'check']
Goal met: True


Adding a restock step should fix the shortage. I manually revised the plan based on the previous failure, allowing all three kits to be packed and verified.

In [8]:
def choose_next(observation, target):
    if observation is None:
        return {"action": "inspect", "quantity": 0}

    state = observation["state"]
    remaining = target - state["packed"]

    if remaining == 0:
        return {"action": "check", "quantity": 0}

    if remaining < 0:
        return None

    stock = state["inventory"]

    if stock["folders"] < remaining:
        return None

    shortage = max(0, remaining - stock["labels"])

    if shortage > stock["spares"]:
        return None

    if shortage:
        return {"action": "restock", "quantity": shortage}

    return {"action": "assemble", "quantity": remaining}

observed = execute({"action": "inspect", "quantity": 0},
                   deepcopy(initial), target)

print(choose_next(observed, target))

{'action': 'restock', 'quantity': 1}


The selector only restocks when labels are missing. If there are not enough spares, it returns None. This allows the program to choose its next action based on the inventory.

In [9]:
def run_agent(start, target, max_steps=4, selector=choose_next):
    if type(target) is not int or not 1 <= target <= 6:
        raise ValueError("target must be an integer from 1 to 6")

    if type(max_steps) is not int or not 1 <= max_steps <= 20:
        raise ValueError("max_steps must be an integer from 1 to 20")

    state, trace = deepcopy(start), []
    observation = None

    if goal_met(state, target):
        return {"status": "success", "state": state, "trace": trace}

    status = "step_limit"

    for _ in range(max_steps):
        proposal = selector(deepcopy(observation), target)

        if proposal is None:
            status = "blocked"
            break

        observation = execute(proposal, state, target)
        trace.append({"proposal": deepcopy(proposal),
                      "observation": observation})

        if not observation["ok"]:
            status = "action_error"
            break

        if goal_met(state, target):
            status = "success"
            break

    return {"status": status, "state": state, "trace": trace}

print("Controller ready.")

Controller ready.


If the goal is verified on the fourth attempt, it should count as success. The controller limits the number of attempts and prevents the program from running forever.

In [10]:
result = run_agent(initial, target)

for n, event in enumerate(result["trace"], start=1):
    proposal = event["proposal"]
    observation = event["observation"]
    print(n, proposal["action"], proposal["quantity"],
          observation["note"])

print("Status:", result["status"])
print("Final state:", result["state"])
print("Initial labels:", initial["inventory"]["labels"])

1 inspect 0 Stock inspected.
2 restock 1 Moved 1 spare label(s).
3 assemble 3 Assembled 3 kit(s).
4 check 0 Count check: True
Status: success
Final state: {'inventory': {'folders': 0, 'labels': 0, 'spares': 0}, 'packed': 3, 'verified': True}
Initial labels: 2


The controller should complete the goal in four attempts. All three kits are packed and verified, and the original inventory remains unchanged.

In [11]:
short_run = run_agent(initial, target, max_steps=3)

no_spares = deepcopy(initial)
no_spares["inventory"]["spares"] = 0
blocked_run = run_agent(no_spares, target)

def repeat_inspect(observation, target):
    return {"action": "inspect", "quantity": 0}

repeat_run = run_agent(initial, target, max_steps=2,
                       selector=repeat_inspect)

print("Short:", short_run["status"], short_run["state"]["packed"],
      short_run["state"]["verified"])

print("No spares:", blocked_run["status"], len(blocked_run["trace"]))

print("Repeated inspection:", repeat_run["status"],
      len(repeat_run["trace"]))

Short: step_limit 3 False
No spares: blocked 1
Repeated inspection: step_limit 2


Packing three kits without verification should not count as success. Step_limit means the allowed attempts ran out. Blocked means the selector cannot continue. Repeated inspection does not make progress.

In [12]:
new_case = {
    "inventory": {"folders": 4, "labels": 3, "spares": 1},
    "packed": 0,
    "verified": False
}

new_result = run_agent(new_case, 4)

def claim_success(observation, target):
    return {"action": "announce_success", "quantity": 0}

claim = run_agent(initial, target, selector=claim_success)

print("New case:", new_result["status"], new_result["state"]["packed"],
      len(new_result["trace"]))

print("Unsupported claim:", claim["status"], claim["state"]["packed"],
      claim["state"]["verified"])

New case: success 4 4
Unsupported claim: action_error 0 False


The new case should complete four kits in four attempts. Success means the goal was packed and verified. Action_error means an invalid action was rejected. The selector cannot claim success without completing the goal.

In [13]:
practice = run_agent(initial, 2, max_steps=3)

print("Status:", practice["status"])
print("Actions:", [e["proposal"]["action"] for e in practice["trace"]])
print("Final state:", practice["state"])

low_folders = deepcopy(initial)
low_folders["inventory"]["folders"] = 1

blocked_practice = run_agent(low_folders, 2, max_steps=3)

print("Low folders:", blocked_practice["status"])
print("Final state:", blocked_practice["state"])

Status: success
Actions: ['inspect', 'assemble', 'check']
Final state: {'inventory': {'folders': 1, 'labels': 0, 'spares': 1}, 'packed': 2, 'verified': True}
Low folders: blocked
Final state: {'inventory': {'folders': 1, 'labels': 2, 'spares': 1}, 'packed': 0, 'verified': False}


The two-kit goal succeeds because the kits are verified on the third attempt. The second test is blocked because there is only one folder available. Getting another folder would fix this problem. Checking the packed count does not prove that the labels have the correct names. The checker would need to compare each label against a list of expected names. These tests show how rule-based planning works, but they do not measure actual LLM performance.